In [0]:
bronze_stream = spark.readStream \
    .format("delta") \
    .table("clickstream_catalog.bronze.raw_events")

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("user_id", StringType()),
    StructField("session_id", StringType()),
    StructField("event_type", StringType()),
    StructField("product_id", StringType()),
    StructField("category", StringType()),
    StructField("price", DoubleType()),
    StructField("timestamp", StringType())  # parse as string first, cast below
])

silver_df = bronze_stream \
    .withColumn("parsed", from_json(col("raw_event_json"), event_schema)) \
    .select(
        col("parsed.event_id"),
        col("parsed.user_id"),
        col("parsed.session_id"),
        col("parsed.event_type"),
        col("parsed.product_id"),
        col("parsed.category"),
        col("parsed.price"),
        col("parsed.timestamp").cast(TimestampType()).alias("event_timestamp"),
        col("kafka_timestamp")
    ) \
    .filter(col("event_id").isNotNull())  # drop malformed/unparseable records

In [0]:
silver_checkpoint = "abfss://silver@itsmystorage.dfs.core.windows.net/clickstream_catalog/_checkpoints/silver_write/"

silver_query = silver_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", silver_checkpoint) \
    .outputMode("append") \
    .toTable("clickstream_catalog.silver.clean_events")

In [0]:
display(spark.sql("SELECT * FROM clickstream_catalog.silver.clean_events ORDER BY event_timestamp DESC LIMIT 10"))

event_id,user_id,session_id,event_type,product_id,category,price,event_timestamp,kafka_timestamp
83ace17e-0e03-4c31-9d58-f2ba4bc80c54,user_70,session_1378,page_view,product_2,toys,null,2026-07-08T07:42:18.294865Z,2026-07-08T07:42:18.284Z
08cdeba2-198b-4c43-bf8a-92dfa087aebd,user_187,session_977,add_to_cart,product_47,sports,119.62,2026-07-08T07:42:17.261806Z,2026-07-08T07:42:17.247Z
2855f2c2-adaf-4328-b967-e3adf3eef2ac,user_312,session_436,page_view,product_103,home,null,2026-07-08T07:42:16.696434Z,2026-07-08T07:42:16.692Z
6fb99b3b-c5f6-4f88-a8f3-b070b8be4378,user_113,session_875,checkout_start,product_140,fashion,495.42,2026-07-08T07:42:15.207388Z,2026-07-08T07:42:15.207Z
e9f8da17-90c7-4886-8388-500b46e9991d,user_38,session_569,page_view,product_161,toys,null,2026-07-08T07:42:13.291574Z,2026-07-08T07:42:13.283Z
3a466106-3a98-4d5b-8aa1-58130706b7c0,user_26,session_554,page_view,product_135,toys,null,2026-07-08T07:42:12.549101Z,2026-07-08T07:42:12.544Z
7107e718-5be8-483c-bd3a-3225dc635f8a,user_53,session_919,add_to_cart,product_187,toys,306.88,2026-07-08T07:42:12.002804Z,2026-07-08T07:42:12.001Z
e3e3c5ec-884a-4a5a-8d1e-ff4b75d23389,user_250,session_785,checkout_start,product_182,electronics,230.06,2026-07-08T07:42:11.384642Z,2026-07-08T07:42:11.378Z
c55a0020-7881-4d5a-a17e-2f50b5b86764,user_58,session_772,page_view,product_21,beauty,null,2026-07-08T07:42:10.806433Z,2026-07-08T07:42:10.798Z
d54f1155-1471-4b8c-bfbc-65b0e334f3ad,user_312,session_168,purchase,product_65,fashion,45.86,2026-07-08T07:42:09.330168Z,2026-07-08T07:42:09.324Z
